In [3]:
import duckdb
import pandas as pd

In [123]:
# DuckDB is an analytical SQL Engine
# It can execute SQL directly against files such as csv, excel, json, parquet

# CSV
#  ↓
# DuckDB
#  ↓
# SQL
#  ↓
# Result

In [124]:
df = pd.read_csv("online_retail.csv" , parse_dates=["InvoiceDate"]) 

In [125]:
conn = duckdb.connect()
conn.register("retail" , df)

In [127]:
# To know the schema
conn.execute("""
DESCRIBE SELECT *
FROM retail
""").df()

,column_name,column_type,null,key,default,extra
0,InvoiceNo,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP_NS,YES,None,None,None
5,UnitPrice,DOUBLE,YES,None,None,None
6,CustomerID,DOUBLE,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [25]:
conn.execute("SELECT * FROM retail LIMIT 5").df()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [30]:
result = conn.execute("SELECT StockCode, Quantity ,UnitPrice  FROM retail LIMIT 15").fetchall()

In [31]:
for r in result:
    print(r)

('85123A', 6, 2.55)
('71053', 6, 3.39)
('84406B', 8, 2.75)
('84029G', 6, 3.39)
('84029E', 6, 3.39)
('22752', 2, 7.65)
('21730', 6, 4.25)
('22633', 6, 1.85)
('22632', 6, 1.85)
('84879', 32, 1.69)
('22745', 6, 2.1)
('22748', 6, 2.1)
('22749', 8, 3.75)
('22310', 6, 1.65)
('84969', 6, 4.25)


## Filtering with WHERE

In [37]:
conn.execute("""
SELECT InvoiceNo,StockCode,Quantity FROM retail
WHERE Quantity > 20
LIMIT 10
""").fetchdf()

,InvoiceNo,StockCode,Quantity
0,536367,84879,32
1,536370,22728,24
2,536370,22727,24
3,536370,21883,24
4,536370,10002,48
5,536370,21791,24
6,536370,22326,24
7,536370,22629,24
8,536370,22659,24
9,536370,22631,24


In [48]:
conn.execute("""
SELECT CustomerID,UnitPrice,Country
FROM retail
WHERE UnitPrice > 20.0
AND Quantity > 30
LIMIT 10
""").fetchdf()

,CustomerID,UnitPrice,Country
0,NaN,25.49,United Kingdom
1,18102.0,32.69,United Kingdom
2,NaN,25.49,United Kingdom
3,NaN,25.49,United Kingdom
4,NaN,25.49,United Kingdom
5,15098.0,649.50,United Kingdom
6,NaN,20.79,United Kingdom
7,18102.0,20.80,United Kingdom


In [49]:
conn.execute("""
SELECT CustomerID,UnitPrice,Country
FROM retail
WHERE UnitPrice > 100.0
OR Quantity > 30
LIMIT 10
""").fetchdf()

,CustomerID,UnitPrice,Country
0,13047.0,1.69,United Kingdom
1,12583.0,0.85,France
2,12583.0,0.65,France
3,13748.0,2.55,United Kingdom
4,15100.0,10.95,United Kingdom
5,15291.0,3.45,United Kingdom
6,15291.0,2.55,United Kingdom
7,14688.0,0.42,United Kingdom
8,14688.0,1.25,United Kingdom
9,14688.0,0.38,United Kingdom


## Sorting

In [58]:
conn.execute("""
SELECT Country , Quantity
FROM retail
ORDER BY Quantity DESC
LIMIT 5
""").fetchdf()

,Country,Quantity
0,United Kingdom,80995
1,United Kingdom,74215
2,United Kingdom,12540
3,United Kingdom,5568
4,United Kingdom,4800


In [60]:
conn.execute("""
SELECT Country , Quantity
FROM retail
ORDER BY Quantity ASC
LIMIT 5
""").fetchdf()

,Country,Quantity
0,United Kingdom,-80995
1,United Kingdom,-74215
2,United Kingdom,-9600
3,United Kingdom,-9600
4,United Kingdom,-9360


## Aggregate

In [62]:
#Aggregate functions
# COUNT()
# SUM()
# AVG()
# MIN()
# MAX()

In [66]:
conn.execute("SELECT SUM(Quantity) FROM retail").fetchall()

[(5176450,)]

In [77]:
conn.execute("""
SELECT
COUNT(Quantity) AS Row_No,
SUM(Quantity) AS Sum,
AVG(Quantity) AS Average,
MIN(Quantity) AS Min,
MAX(Quantity) AS Max,
FROM retail
""").fetchdf()

,Row_No,Sum,Average,Min,Max
0,541909,5176450.0,9.55225,-80995,80995


 # Application 1
 

In [78]:
conn.execute("SELECT * FROM retail LIMIT 1").fetchdf()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom


In [82]:
#To get the revenue for each InvoiceNo
conn.execute("""
SELECT 
    InvoiceNo, 
    Quantity ,
    UnitPrice, 
    Country,
    UnitPrice * Quantity AS Revenue
FROM retail
LIMIT 10
""").fetchdf()

,InvoiceNo,Quantity,UnitPrice,Country,Revenue
0,536365,6,2.55,United Kingdom,15.30
1,536365,6,3.39,United Kingdom,20.34
2,536365,8,2.75,United Kingdom,22.00
3,536365,6,3.39,United Kingdom,20.34
4,536365,6,3.39,United Kingdom,20.34
5,536365,2,7.65,United Kingdom,15.30
6,536365,6,4.25,United Kingdom,25.50
7,536366,6,1.85,United Kingdom,11.10
8,536366,6,1.85,United Kingdom,11.10
9,536367,32,1.69,United Kingdom,54.08


In [86]:
conn.execute("SELECT SUM(UnitPrice * Quantity) AS Total_revenue FROM retail").fetchdf()

,Total_revenue
0,9.747748e+06


## Group By

In [87]:
# GROUP BY creates groups---> collapses rows to form groups


In [91]:
conn.execute("""
SELECT
    Country,
    SUM(UnitPrice * Quantity) AS Revenue
FROM retail
GROUP BY Country
LIMIT 10
""").fetchdf()

,Country,Revenue
0,France,197403.90
1,Germany,221698.21
2,EIRE,263276.82
3,Portugal,29367.02
4,Sweden,36595.91
5,Singapore,9120.39
6,Saudi Arabia,131.17
7,USA,1730.92
8,Canada,3666.38
9,Norway,35163.46


In [ ]:
# GROUP BY defines the groups. Aggregations summarize those groups.

# Application 2

In [109]:
#Finding the most AND min Amount per countries
conn.execute("""
SELECT
    Country,
    MAX(Quantity) AS Max_Quantity_purchased_per_Invoice,
    MIN(Quantity) AS Min_Quantity_purchased_per_Invoice    
FROM retail
GROUP BY Country
ORDER BY MIN_Quantity_purchased_per_Invoice ASC
LIMIT 10
""").fetchdf()

,Country,Max_Quantity_purchased_per_Invoice,Min_Quantity_purchased_per_Invoice
0,United Kingdom,80995,-80995
1,Japan,2040,-624
2,Netherlands,2400,-480
3,EIRE,1440,-288
4,Germany,600,-288
5,Spain,360,-288
6,France,912,-250
7,Sweden,768,-240
8,Australia,1152,-120
9,Switzerland,288,-120


In [116]:
#checcking if truly there are negative quantiy
conn.execute("""
SELECT
    InvoiceNo,
    Quantity
FROM retail
WHERE Quantity < 0
ORDER BY Quantity ASC
LIMIT 10
""").df()

,InvoiceNo,Quantity
0,C581484,-80995
1,C541433,-74215
2,556691,-9600
3,556690,-9600
4,C536757,-9360
5,556687,-9058
6,546152,-5368
7,573596,-4830
8,566768,-3667
9,565304,-3167


In [122]:
# Close the DuckDB connection.
conn.close()